# Get Past 1 Month Prediction

**This notebook fetches data of past 1 month, gathers model predictions, and saves to predictions.json as `history`**


In [2]:
import requests
import pandas as pd
import json
import numpy as np

# Get Weather Data


Geo coords for Dhaka, Chittagong, and Patuakhali


In [3]:
# tuple as (lat, lon)
dhk_coords = (23.7104, 90.40744)
cht_coords = (22.3384, 91.83168)
pat_coords = (22.36833, 90.3458)

In [ ]:
from datetime import datetime, timedelta

# start_date = datetime.today() - timedelta(days=31)  # 1 month ago date
# start_date = start_date.strftime("%Y-%m-%d")
# end_date = datetime.today() - timedelta(days=1)  # yesterday date
# end_date = end_date.strftime("%Y-%m-%d")

start_date = "2021-04-01"
end_date = "2025-09-01"

In [5]:
start_date

'2021-04-01'

In [6]:
def get_city_weather(coords, start_date, end_date):
    weather_api_url = f"https://historical-forecast-api.open-meteo.com/v1/forecast?latitude={coords[0]}&longitude={coords[1]}&start_date={start_date}&end_date={end_date}&daily=relative_humidity_2m_mean,temperature_2m_max,temperature_2m_min,temperature_2m_mean,rain_sum,sunshine_duration"
    # weather_api_url = f"https://historical-forecast-api.open-meteo.com/v1/forecast?latitude={coords[0]}&longitude={coords[1]}&start_date={start_date}&end_date={end_date}%hourly=temperature_2m"
    response = requests.get(weather_api_url)

    try:
        response.raise_for_status()
        data = response.json()

        return data
    except requests.exceptions.RequestException as e:
        print(e)

In [7]:
dhaka_data = get_city_weather(dhk_coords, start_date, end_date)
# dhaka_data

In [8]:
city_weather_df = pd.DataFrame(
    columns=[
        "Date",  # Remove Date before passed to model
        "Rainfall",
        "Sunshine",
        "Humidity",
        "Temp_mean",
        "Temp_max",
        "Temp_min",
        "Year",
        "Month",
        # "loadshed_prev",
        # "generation_prev",
    ]
)
city_weather_df

,Date,Rainfall,Sunshine,Humidity,Temp_mean,Temp_max,Temp_min,Year,Month


In [9]:
daily_data = dhaka_data["daily"]

city_weather_df["Date"] = daily_data["time"]
city_weather_df["Date"] = pd.to_datetime(city_weather_df["Date"], format="%Y-%m-%d")
city_weather_df["Year"] = city_weather_df["Date"].dt.year
city_weather_df["Month"] = city_weather_df["Date"].dt.month

city_weather_df["Rainfall"] = daily_data["rain_sum"]
city_weather_df["Sunshine"] = daily_data["sunshine_duration"]
city_weather_df["Sunshine"] = city_weather_df["Sunshine"] / (60 * 60)
city_weather_df["Humidity"] = daily_data["relative_humidity_2m_mean"]

city_weather_df["Temp_mean"] = daily_data["temperature_2m_mean"]
city_weather_df["Temp_max"] = daily_data["temperature_2m_max"]
city_weather_df["Temp_min"] = daily_data["temperature_2m_min"]

city_weather_df

,Date,Rainfall,Sunshine,Humidity,Temp_mean,Temp_max,Temp_min,Year,Month
0,2021-04-01,0.0,11.661556,53,31.2,39.6,24.7,2021,4
1,2021-04-02,0.0,11.197658,50,31.1,39.0,25.3,2021,4
2,2021-04-03,0.0,11.706211,32,32.0,39.1,25.3,2021,4
3,2021-04-04,0.0,11.640431,25,31.6,39.4,25.7,2021,4
4,2021-04-05,0.0,11.750733,14,31.5,38.1,24.9,2021,4
...,...,...,...,...,...,...,...,...,...
1610,2025-08-28,0.0,7.762828,85,28.2,31.6,26.2,2025,8
1611,2025-08-29,0.0,8.890681,85,28.5,32.5,26.1,2025,8
1612,2025-08-30,0.0,10.120406,85,28.9,32.1,26.4,2025,8
1613,2025-08-31,0.0,9.986231,85,29.3,32.5,27.0,2025,8


In [10]:
def get_city_weather_df(daily_data):
    city_df = pd.DataFrame(
        columns=[
            "Date",  # Remove Date before passed to model
            "Rainfall",
            "Sunshine",
            "Humidity",
            "Temp_mean",
            "Temp_max",
            "Temp_min",
            "Year",
            "Month",
            # "loadshed_prev",
            # "generation_prev",
        ]
    )

    city_df["Date"] = daily_data["time"]
    city_df["Date"] = pd.to_datetime(city_df["Date"], format="%Y-%m-%d")
    city_df["Year"] = city_df["Date"].dt.year
    city_df["Month"] = city_df["Date"].dt.month

    city_df["Rainfall"] = daily_data["rain_sum"]
    city_df["Sunshine"] = daily_data["sunshine_duration"]
    # Convert seconds to hours
    city_df["Sunshine"] = city_df["Sunshine"] / (60 * 60)
    city_df["Humidity"] = daily_data["relative_humidity_2m_mean"]

    city_df["Temp_mean"] = daily_data["temperature_2m_mean"]
    city_df["Temp_max"] = daily_data["temperature_2m_max"]
    city_df["Temp_min"] = daily_data["temperature_2m_min"]

    return city_df

## Get weather data of all locations


In [11]:
dhk_data = get_city_weather(dhk_coords, start_date, end_date)
dhk_weather_df = get_city_weather_df(dhk_data["daily"])
dhk_weather_df

,Date,Rainfall,Sunshine,Humidity,Temp_mean,Temp_max,Temp_min,Year,Month
0,2021-04-01,0.0,11.661556,53,31.2,39.6,24.7,2021,4
1,2021-04-02,0.0,11.197658,50,31.1,39.0,25.3,2021,4
2,2021-04-03,0.0,11.706211,32,32.0,39.1,25.3,2021,4
3,2021-04-04,0.0,11.640431,25,31.6,39.4,25.7,2021,4
4,2021-04-05,0.0,11.750733,14,31.5,38.1,24.9,2021,4
...,...,...,...,...,...,...,...,...,...
1610,2025-08-28,0.0,7.762828,85,28.2,31.6,26.2,2025,8
1611,2025-08-29,0.0,8.890681,85,28.5,32.5,26.1,2025,8
1612,2025-08-30,0.0,10.120406,85,28.9,32.1,26.4,2025,8
1613,2025-08-31,0.0,9.986231,85,29.3,32.5,27.0,2025,8


In [12]:
cht_data = get_city_weather(cht_coords, start_date, end_date)
cht_weather_df = get_city_weather_df(cht_data["daily"])
cht_weather_df

,Date,Rainfall,Sunshine,Humidity,Temp_mean,Temp_max,Temp_min,Year,Month
0,2021-04-01,0.0,10.767494,78,27.0,32.0,23.0,2021,4
1,2021-04-02,0.0,9.386283,76,26.3,31.5,21.6,2021,4
2,2021-04-03,0.0,11.691586,58,26.9,33.4,20.0,2021,4
3,2021-04-04,0.0,11.712381,64,27.2,34.1,20.2,2021,4
4,2021-04-05,0.0,11.733131,66,27.0,33.0,20.6,2021,4
...,...,...,...,...,...,...,...,...,...
1610,2025-08-28,0.0,7.790697,84,28.0,28.6,26.8,2025,8
1611,2025-08-29,0.0,9.417219,87,27.8,28.4,27.2,2025,8
1612,2025-08-30,0.0,8.796036,86,28.1,28.7,27.0,2025,8
1613,2025-08-31,0.0,10.874611,86,28.5,29.8,27.1,2025,8


In [13]:
pat_data = get_city_weather(pat_coords, start_date, end_date)
pat_weather_df = get_city_weather_df(pat_data["daily"])
pat_weather_df

,Date,Rainfall,Sunshine,Humidity,Temp_mean,Temp_max,Temp_min,Year,Month
0,2021-04-01,0.0,11.649914,67,29.5,37.5,24.6,2021,4
1,2021-04-02,0.0,10.950492,66,28.9,35.3,23.7,2021,4
2,2021-04-03,0.0,11.691586,61,29.5,38.0,23.5,2021,4
3,2021-04-04,1.6,11.097867,38,30.3,38.7,23.6,2021,4
4,2021-04-05,0.0,11.733131,30,30.0,38.2,23.2,2021,4
...,...,...,...,...,...,...,...,...,...
1610,2025-08-28,0.0,6.524497,89,27.8,30.2,26.3,2025,8
1611,2025-08-29,0.0,7.189942,89,27.9,30.6,26.3,2025,8
1612,2025-08-30,0.0,4.428906,90,28.1,30.9,26.3,2025,8
1613,2025-08-31,0.0,8.152969,85,29.4,32.6,26.3,2025,8


In [14]:
all_weather_df = pd.concat([dhk_weather_df, cht_weather_df, pat_weather_df])

all_weather_df = (
    all_weather_df.groupby("Date")
    .agg(
        {
            "Rainfall": "mean",
            "Sunshine": "mean",
            "Humidity": "mean",
            "Temp_mean": "mean",
            "Temp_max": "max",
            "Temp_min": "min",
            "Year": "first",
            "Month": "first",
            # "loadshed_prev": "first",
            # "generation_prev": "first",
        }
    )
    .reset_index()
)

all_weather_df

,Date,Rainfall,Sunshine,Humidity,Temp_mean,Temp_max,Temp_min,Year,Month
0,2021-04-01,0.000000,11.359655,66.000000,29.233333,39.6,23.0,2021,4
1,2021-04-02,0.000000,10.511478,64.000000,28.766667,39.0,21.6,2021,4
2,2021-04-03,0.000000,11.696461,50.333333,29.466667,39.1,20.0,2021,4
3,2021-04-04,0.533333,11.483559,42.333333,29.700000,39.4,20.2,2021,4
4,2021-04-05,0.000000,11.738998,36.666667,29.500000,38.2,20.6,2021,4
...,...,...,...,...,...,...,...,...,...
1610,2025-08-28,0.000000,7.359341,86.000000,28.000000,31.6,26.2,2025,8
1611,2025-08-29,0.000000,8.499281,87.000000,28.066667,32.5,26.1,2025,8
1612,2025-08-30,0.000000,7.781782,87.000000,28.366667,32.1,26.3,2025,8
1613,2025-08-31,0.000000,9.671270,85.333333,29.066667,32.6,26.3,2025,8


In [ ]:
all_weather_df.to_csv("data/weather_data_latest.csv", index=False)